### Imports & Setup

Run the first cell once, then click "Restart Kernel and Run All Cells", (all cells including the first one)

In [72]:
%pip install -r requirements.txt

import os, sys, subprocess
import zipfile
import pandas as pd
import pm4py
from datetime import datetime
import pytz
from logview.utils import LogViewBuilder
from logview.predicate import *
from filter_visualization import query_exploration_icicle, query_breakdown_pie, interactive_icicle, interactive_pie, chart_selecting

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


### Import LogView

In [73]:
if not os.path.exists("logview"):
    subprocess.run(["git", "clone", "https://github.com/fzerbato/logview.git"], check=True)
    print("Cloned logview.")
else:
    print("logview already cloned.")

logview_path = os.path.abspath("logview")

if logview_path not in sys.path:
    sys.path.append(logview_path)
    print(f"Added to sys.path: {logview_path}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "logview/requirements.txt"], check=True)


logview already cloned.


CompletedProcess(args=['c:\\Program Files\\Python310\\python.exe', '-m', 'pip', 'install', '-r', 'logview/requirements.txt'], returncode=0)

### Load and Prepare Event Log  

This notebook uses the **Sepsis Cases Event Log** (Mannhardt, 2016)

[Mannhardt, F. (2016). *Sepsis Cases - Event Log* [Data set]. 4TU.ResearchData. https://doi.org/10.4121/uuid:915d2bfb-7e84-49ad-a286-dc35f063a460].  

This real-life event log contains events of sepsis cases from a hospital. Sepsis is a life threatening condition typically caused by an infection. One case represents the pathway through the hospital. The events were recorded by the ERP (Enterprise Resource Planning) system of the hospital. There are about 1000 cases with in total 15,000 events that were recorded for 16 different activities. Moreover, 39 data attributes are recorded, e.g., the group responsible for the activity, the results of tests and information from checklists. Events and attribute values have been anonymized. The time stamps of events have been randomized, but the time between events within a trace has not been altered. 


In [74]:
import pandas as pd
import pm4py

CASE_ID_COL = 'Case ID'
TIMESTAMP_COL = 'Complete Timestamp'
ACTIVITY_COL = 'Activity'


path_to_log = "./dataset/Sepsis Cases - Event Log.xes.gz"
df = pm4py.read_xes(path_to_log)

df = df.rename(columns={
    "case:concept:name": CASE_ID_COL,
    "time:timestamp": TIMESTAMP_COL,
    "concept:name": ACTIVITY_COL
})

df = df.sort_values([CASE_ID_COL, TIMESTAMP_COL], ignore_index=True)
log = pm4py.format_dataframe(
    df,
    case_id=CASE_ID_COL,
    activity_key=ACTIVITY_COL,
    timestamp_key=TIMESTAMP_COL
)

display(log)


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

,InfectionSuspected,org:group,DiagnosticBlood,DisfuncOrg,SIRSCritTachypnea,Hypotensie,SIRSCritHeartRate,Infusion,DiagnosticArtAstrup,Activity,...,DiagnosticECG,Case ID,Leucocytes,CRP,LacticAcid,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,True,A,True,True,True,True,True,True,True,ER Registration,...,True,A,NaN,NaN,NaN,A,ER Registration,2014-10-22 11:15:41+00:00,0,0
1,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Leucocytes,...,NaN,A,9.6,NaN,NaN,A,Leucocytes,2014-10-22 11:27:00+00:00,1,0
2,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRP,...,NaN,A,NaN,21.0,NaN,A,CRP,2014-10-22 11:27:00+00:00,2,0
3,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LacticAcid,...,NaN,A,NaN,NaN,2.2,A,LacticAcid,2014-10-22 11:27:00+00:00,3,0
4,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ER Triage,...,NaN,A,NaN,NaN,NaN,A,ER Triage,2014-10-22 11:33:37+00:00,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15209,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRP,...,NaN,ZZ,NaN,146.0,NaN,ZZ,CRP,2014-11-14 07:00:00+00:00,15209,1049
15210,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Leucocytes,...,NaN,ZZ,8.3,NaN,NaN,ZZ,Leucocytes,2014-11-14 07:00:00+00:00,15210,1049
15211,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Leucocytes,...,NaN,ZZ,7.7,NaN,NaN,ZZ,Leucocytes,2014-11-16 07:00:00+00:00,15211,1049
15212,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRP,...,NaN,ZZ,NaN,96.0,NaN,ZZ,CRP,2014-11-16 07:00:00+00:00,15212,1049


In [75]:
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"\n {col} ")
    print(unique_vals[:20])
    if len(unique_vals) > 20:
        print(f"{len(unique_vals) - 20} more")



 InfectionSuspected 
[True nan False]

 org:group 
['A' 'B' 'C' 'D' 'E' 'I' '?' 'F' 'O' 'S' 'P' 'R' 'Q' 'V' 'G' 'W' 'M' 'L'
 'J' 'H']
6 more

 DiagnosticBlood 
[True nan False]

 DisfuncOrg 
[True nan False]

 SIRSCritTachypnea 
[True nan False]

 Hypotensie 
[True nan False]

 SIRSCritHeartRate 
[True nan False]

 Infusion 
[True nan False]

 DiagnosticArtAstrup 
[True nan False]

 Activity 
['ER Registration' 'Leucocytes' 'CRP' 'LacticAcid' 'ER Triage'
 'ER Sepsis Triage' 'IV Liquid' 'IV Antibiotics' 'Admission NC'
 'Release A' 'Return ER' 'Admission IC' 'Release B' 'Release E'
 'Release C' 'Release D']

 Age 
[85. nan 75. 60. 90. 80. 55. 70. 50. 65. 35. 45. 25. 30. 20. 40.]

 DiagnosticIC 
[True nan False]

 DiagnosticSputum 
[False nan True]

 DiagnosticLiquor 
[False nan True]

 DiagnosticOther 
[False nan True]

 SIRSCriteria2OrMore 
[True nan False]

 DiagnosticXthorax 
[True nan False]

 SIRSCritTemperature 
[True nan False]

 Complete Timestamp 
<DatetimeArray>
['2014-10-22 1

### Instantiating a LogView object ###
The LogViewBuilder class allows instantiating a LogView object, which serves as the central interface for creating LogView instances. It provides a single point of access for interacting with the different framework components.

In [76]:
# Build LogView
log_view = LogViewBuilder.build_log_view(log)

### Exploration 

For the first analysis, I approached the dataset as a non-medical analyst, aiming to explore factors that might influence how long patients remain in the ER and whether certain variables interact with one another. Specifically, I examined:  

- Whether the patient was over 65 years of age  
- Whether the patient received IV antibiotics at any point  
- Whether the patient received an X-ray  

Based on these considerations, I applied the following filters:  

In [77]:
# Elderly patients (Age ≥ 65)
query_age = Query('Elderly', GreaterEqualToConstant('Age', 65))
rs_age, complement_age = log_view.evaluate_query('rs_Elderly', log, query_age)

# Patients who received IV Antibiotics
query_abx = Query('Elderly_with_Antibiotics', EqToConstant('Activity', 'IV Antibiotics'))
rs_abx, complement_abx = log_view.evaluate_query('rs_Elderly_with_Antibiotics', rs_age, query_abx)

# Patients who also had a Chest X-ray
query_xray = Query('Elderly_with_Antibiotics_and_XRay', EqToConstant('DiagnosticXthorax', True))
rs_xray, complement_xray = log_view.evaluate_query('rs_Elderly_with_Antibiotics_and_XRay', rs_abx, query_xray)

In [78]:
summary = log_view.get_summary()

+----+-----------------------------+-----------------------------------+--------------------------------------+----------+
|    | source_log                  | query                             | result_set                           | labels   |
|----+-----------------------------+-----------------------------------+--------------------------------------+----------|
|  0 | initial_source_log          | Elderly                           | rs_Elderly                           | []       |
|  1 | rs_Elderly                  | Elderly_with_Antibiotics          | rs_Elderly_with_Antibiotics          | []       |
|  2 | rs_Elderly_with_Antibiotics | Elderly_with_Antibiotics_and_XRay | rs_Elderly_with_Antibiotics_and_XRay | []       |
+----+-----------------------------+-----------------------------------+--------------------------------------+----------+
+----+-----------------------------------+------------------------------------+
|    | query                             | predicates      

When examining the visualization results, it becomes clear that, as expected, elderly patients form a substantial proportion of ER visitors and generally remain in the ER for longer periods. Among these elderly patients, roughly three-quarters received IV antibiotics, compared to about two-thirds of patients younger than 65. Receiving IV antibiotics significantly increased the length of stay, with the effect being more pronounced in elderly patients than in younger ones.  

In contrast, X-rays were common across all groups but did not appear to substantially extend ER stay durations. Similarly, the patterns for time between events and the total number of events mirrored those for overall duration: both increased with the administration of IV antibiotics, but showed no additional increase when X-rays were involved.  


In [79]:
query_exploration_icicle('rs_Elderly_with_Antibiotics_and_XRay', log_view, metric='avg_case_duration_seconds', details=False)
query_exploration_icicle('rs_Elderly_with_Antibiotics_and_XRay', log_view, metric='avg_events_per_case', details=False)
query_exploration_icicle('rs_Elderly_with_Antibiotics_and_XRay', log_view, metric='avg_time_between_events', details=False)

### Testing Assumptions  

Next, I wanted to test some assumptions I had about the log.  
Specifically, I expected there to be a strong overlap between patients with a suspected infection, those who underwent blood diagnostics (either as a reason to suspect infection or as a consequence of it), and those who received an infusion.  

To examine this, I applied the following filters:  

In [80]:
# Patients with suspected infection
query_infection = Query('Suspected_Infection', EqToConstant('InfectionSuspected', True))
rs_infection, complement_infection = log_view.evaluate_query('rs_Suspected_Infection', log, query_infection)

# Patients who had a blood diagnostic
query_blood = Query('Infection_with_BloodDiagnostic', EqToConstant('DiagnosticBlood', True))
rs_blood, complement_blood = log_view.evaluate_query('rs_Infection_with_BloodDiagnostic', rs_infection, query_blood)

# Patients who received an infusion
query_infusion = Query('Infection_with_BloodDiagnostic_and_Infusion', EqToConstant('Infusion', True))
rs_infusion, complement_infusion = log_view.evaluate_query('rs_Infection_with_BloodDiagnostic_and_Infusion', rs_blood, query_infusion)

summary = log_view.get_summary()

+----+-----------------------------------+---------------------------------------------+------------------------------------------------+----------+
|    | source_log                        | query                                       | result_set                                     | labels   |
|----+-----------------------------------+---------------------------------------------+------------------------------------------------+----------|
|  0 | initial_source_log                | Elderly                                     | rs_Elderly                                     | []       |
|  1 | rs_Elderly                        | Elderly_with_Antibiotics                    | rs_Elderly_with_Antibiotics                    | []       |
|  2 | rs_Elderly_with_Antibiotics       | Elderly_with_Antibiotics_and_XRay           | rs_Elderly_with_Antibiotics_and_XRay           | []       |
|  3 | initial_source_log                | Suspected_Infection                         | rs_Suspected_Infe

When inspecting the visualizations, my initial assumption appears to hold. Most cases in the log are either true for all three filters or false for all three. For example, patients without a suspected infection stayed in the ER for about half as long (16 days) compared to those with a suspected infection.  

An unexpected finding emerged in the group where infection was suspected but no blood diagnostics were performed. Although this group is relatively small, their outcomes differed dramatically depending on whether they received an infusion: patients who did receive an infusion had an average stay of 81 days, while those who did not stayed only about 9 days. This big difference raises questions about what factors might explain such a large gap. Interestingly, these long-duration cases have on average only about five more events, but the time between events is much longer.

In [81]:
query_exploration_icicle('rs_Infection_with_BloodDiagnostic_and_Infusion', log_view, metric='avg_case_duration_seconds', details=False)
query_exploration_icicle('rs_Infection_with_BloodDiagnostic_and_Infusion', log_view, metric='avg_events_per_case', details=False)
query_exploration_icicle('rs_Infection_with_BloodDiagnostic_and_Infusion', log_view, metric='avg_time_between_events', details=False)

### ICU Admissions

Next, I was interested in exploring ICU admissions following ER stays.
The first filter I applied checked whether patients were admitted to the ICU at any point during their stay. I then investigated whether this outcome was associated with CRP levels (C-reactive protein, a marker of inflammation, with 100 mg/L taken as a cutoff for severity) and with whether patients received IV antibiotics during their treatment.

In [ ]:
# Patients with ICU Admission
query_ic = Query('Admission_IC', EqToConstant('Activity', 'Admission IC'))
rs_ic, complement_ic = log_view.evaluate_query('rs_Admission_IC', log, query_ic)

# Patients with CRP ≥ 100
query_crp = Query('ICU_with_High_CRP', GreaterEqualToConstant('CRP', 100))
rs_crp, complement_crp = log_view.evaluate_query('rs_ICU_with_High_CRP', rs_ic, query_crp)

# Patients who also received IV Antibiotics
query_abx = Query('ICU_HighCRP_with_Antibiotics', EqToConstant('Activity', 'IV Antibiotics'))
rs_abx, complement_abx = log_view.evaluate_query('rs_ICU_HighCRP_with_Antibiotics', rs_crp, query_abx)

# Summary of results
summary = log_view.get_summary()



+----+-----------------------------------+---------------------------------------------+------------------------------------------------+----------+
|    | source_log                        | query                                       | result_set                                     | labels   |
|----+-----------------------------------+---------------------------------------------+------------------------------------------------+----------|
|  0 | initial_source_log                | Elderly                                     | rs_Elderly                                     | []       |
|  1 | rs_Elderly                        | Elderly_with_Antibiotics                    | rs_Elderly_with_Antibiotics                    | []       |
|  2 | rs_Elderly_with_Antibiotics       | Elderly_with_Antibiotics_and_XRay           | rs_Elderly_with_Antibiotics_and_XRay           | []       |
|  3 | initial_source_log                | Suspected_Infection                         | rs_Suspected_Infe

The results show that only a small proportion of ER cases lead to ICU admission. These cases are characterized by longer durations and a higher number of events per case, while the average time between events remains similar to non-ICU cases. This suggests that time between events is not a strong indicator of ICU admission. Patients admitted to the ICU more often had elevated CRP values, and most of them also received IV antibiotics.

An interesting pattern appeared among ICU cases with lower CRP values: those who received antibiotics tended to have relatively short stays, while those without antibiotics had extremely long stays. However, this group is a small subset of the log, which means these results should be interpreted with caution.

In [83]:
query_exploration_icicle('rs_ICU_HighCRP_with_Antibiotics', log_view, metric='avg_case_duration_seconds', details=False)
query_exploration_icicle('rs_ICU_HighCRP_with_Antibiotics', log_view, metric='avg_events_per_case', details=False)
query_exploration_icicle('rs_ICU_HighCRP_with_Antibiotics', log_view, metric='avg_time_between_events', details=False)